# Project K, codec-loss control: MiniCPM-o 2.6

**Why this notebook exists.** Every Project K run so far fed
the models sound that had passed through a lossy codec: Pilot
One at AAC 96k, Pilot Two and the swap files with the swap
build recipe. A lossy codec throws away quiet detail by
design. So a fair objection stands against the whole record:
when a model missed a quiet sound, maybe the file lost it
before the model ever had the chance. This notebook is the
control that answers the objection. It asks the same model
the same questions about the same sounds, twice over: once
with the sound inside an MP4, decoded the way every earlier
run decoded it, and once with the same sound handed over as
an uncompressed WAV waveform that never passed through the
codec.

**What this notebook does.** Seven sounds, each exactly 10
seconds, each in two arms, each asked two ways, two repeats:
56 asks.

- **The sounds.** A 1000 Hz tone at the reference level
  (burst RMS -20 dBFS); the same tone 20 dB quieter; three
  short sharp clicks at the quiet level; the spoken sentence
  (the swap study's own speech file, so this control ties
  back to it); the quiet tone mixed under the matched vacuum
  bed; a quiet two-second noise burst, the sizzle-shaped
  case; and true silence.
- **Arm A, the file way.** The sound inside an MP4 built
  with the swap recipe: the same still picture, the native
  AAC encoder. The recipe sets no bitrate and the encoder's
  rate follows the content; this battery's speech file
  lands at about 38 kbps, matching the shipped swap speech
  file on the same soundtrack. Arm A reproduces the recipe,
  which is what every earlier run actually rode.
- **Arm B, the raw way.** The same sound as a WAV master.
  The runner loads the waveform straight from the WAV and
  builds the same one-second chat units as Arm A, each unit
  carrying the swap still and the matching second of that
  waveform. The audio array never passes through an MP4.
- **The forms.** The open question is the swap study's P3,
  word for word. The yes/no question is the Series Two form,
  "Is there the sound of X in this clip? Answer yes or no.",
  with X matched to the sound. This model's chat door
  exposes no yes score (settled from the code in Series
  Two), so the yes/no rows are words only, as in its Series
  Two run.

**The gates are files, not answers.** There is no sanity
question in this notebook. A model-answer gate once
misfired on a missing file and stopped a healthy run, so
here the gates are the file and feed assertions, run before
the battery over every file: each file is decoded through
the very path its arm will ride, and the runner prints its
path, SHA-256, duration, and per-second RMS, then asserts
the duration is 10.0 seconds within half a second, asserts
the track is not silent unless the sound is declared
silence (and that declared silence really is zeros), and
for Arm B asserts the waveform handed toward the model
hashes equal to a direct decode of the same WAV in the same
session. Any failure stops the run with exit code 7 before
a single ask. The same proof then repeats per ask, before
each answer, on the exact decoded track the chat receives.

**Resume.** The battery skips any ask already present in
`codec_minicpm_results.csv`, so a stopped session can be
continued by running Step 4 again.

**How to read the result** (fixed in advance, from the
codec control plan):

- Caught raw, missed in AAC: the file did it. That sound's
  earlier misses get re-labeled.
- Missed in both: the model did it; the codec objection
  dies for that sound.
- Caught in both: codec loss does not explain that sound.
- Caught in AAC, missed raw: a build bug. Stop and inspect
  before believing anything.

**Where it runs.** On the paid Colab account (kobbytti),
like the other MiniCPM runs, under its sealed private
environment. **How to run it.** Run the cells from the top,
in order. Step 1 builds the private environment and proves
the stack. Step 2 gets the fifteen battery files. Step 3
writes the runner and runs its check mode (proof, loading
ladder, file assertions). Step 4 runs the battery and saves
`codec_minicpm_results.csv`.

**The one rule.** We claim only what we ran: these sounds,
this model, these two feed paths, this run.

## Step 1 · Setup: a private environment for the model's stack

The next cell does four things:

1. Installs `uv`, a fast environment and package tool, if this machine
   does not already have it. That is the only thing it adds to Colab's
   own Python, and it is a tool, not part of the model's stack.
2. Creates a private virtual environment at `/content/minicpm-venv`,
   on Python 3.12, the same Python generation Colab itself runs. The
   model's stack was proven to install cleanly on Python 3.12, so the
   private environment keeps 3.12 and changes only who owns it.
3. Installs the model's stack inside that environment, in two passes.
   First, torch and torchaudio pinned to 2.8.0 from PyTorch's CUDA 12.6
   shelf: that is the exact build this runtime already runs, so the
   chip driver match is the proven one, and the installer cannot drag
   in a multi-gigabyte CUDA 13 torch the driver may not support (left
   floating, the resolver does exactly that). Second, the model card's
   own pin list, unchanged: transformers 4.44.2, accelerate 0.34.2,
   bitsandbytes 0.46.1, librosa 0.10.2.post1 (a newer patch than the
   card lists, because the card's version predates Colab's current
   numpy), soundfile, vocos, and vector-quantize-pytorch. Pillow rides
   along because the model's code and the battery both need it and
   nothing in the pin list pulls it in. vocos and
   vector-quantize-pytorch are parts the model's code imports for its
   speech side; they are installed even though this test never asks
   the model to speak. One more install rides after those two passes:
   flash-attn 2.8.3, the attention library the model's own modeling
   file imports. Transformers checks that file's imports before it
   loads a single weight, so without flash-attn the model cannot
   even start. Colab's stock environment carries it already, which
   is why the model card never lists it; the private environment
   gets its own copy as the official prebuilt wheel for this exact
   stack (Python 3.12, torch 2.8, CUDA 12, cxx11 ABI), pinned by its
   full download address, never a build from source.
4. Proves the stack, in a fresh process under the environment's own
   Python. The proof prints the Python version, the torch version, the
   transformers version and the file it loads from, and it imports
   `WHISPER_ATTENTION_CLASSES` from the transformers whisper module,
   the exact symbol the model's own code asks for and newer
   transformers versions removed, and it imports flash_attn and
   prints its version. A version number alone is not proof anymore;
   the imports are. If any line of the proof is wrong, Step 1 stops
   on the spot and the run goes no further.

Nothing in this cell installs over, downgrades, or removes anything
Colab shipped. The two failed runs of this notebook both died in that
fight; this design simply refuses to have it.

The install downloads a few gigabytes (torch is the big one) and takes
several minutes. That is normal. Every command is printed before it
runs, and the installers stream their output into the cell, so nothing
about this step happens off the record.

In [ ]:
# --- Setup: build the private environment and prove the stack inside it --------
# This cell never installs the model's stack into Colab's Python. Two
# earlier runs of this notebook died doing exactly that: the pinned
# transformers never landed where the session reads. The stack now
# lives in a private virtual environment at /content/minicpm-venv, and
# Step 1 ends by proving, in a fresh process under that environment's
# Python, that the pinned versions and the one symbol the model's code
# needs are really there.
import os, sys, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VENV = "/content/minicpm-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

def run_stream(cmd):
    print("+", " ".join(cmd))
    proc = subprocess.run(cmd)
    print("  exit code:", proc.returncode)
    return proc.returncode

# 1. uv: the environment tool. The only addition to Colab's own Python.
if shutil.which("uv") is None:
    print("uv not found. Installing it (a tool only, not the model stack).")
    rc = run_stream([sys.executable, "-m", "pip", "install", "uv"])
    if rc != 0 or shutil.which("uv") is None:
        print()
        print("!" * 72)
        print("STOP. uv could not be installed, so the private environment")
        print("cannot be built. Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: uv install failed")
print("uv:", shutil.which("uv"))

# 2. The private environment itself, on Python 3.12. The card's stack
# was proven to install cleanly on Python 3.12 (a full clean resolution
# of the exact pin list), so the private environment keeps 3.12 and
# changes only who owns the environment: this run, not Colab.
if not os.path.exists(VENV_PY):
    rc = run_stream(["uv", "venv", "--python", "3.12", VENV])
    if rc != 0:
        print()
        print("!" * 72)
        print("STOP. The private environment could not be created.")
        print("Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: venv creation failed")
ver = subprocess.run([VENV_PY, "--version"], capture_output=True, text=True)
print("venv python:", (ver.stdout or ver.stderr).strip())

# 3a. torch first, pinned to the runtime's own proven build. Colab's
# pip list on the failed runs showed torch 2.8.0+cu126: that is the
# build this chip and driver already run. Left floating, the resolver
# picks a newest torch built on CUDA 13 and downloads gigabytes of
# driver libraries the runtime may not support. Pinning it is not a
# change to the model card's list; the card leaves torch to the
# machine, and this makes the machine's proven choice explicit.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "torch==2.8.0", "torchaudio==2.8.0",
                 "--index-url", "https://download.pytorch.org/whl/cu126"])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The torch install inside the private environment failed.")
    print("The installer's own output is above. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: torch install failed in venv")

# 3b. The model card's pin list, unchanged, in one resolver pass. No
# force-reinstall: the environment is empty and private, so there is
# nothing to force against. Pillow is added because the model's code
# and the battery import it and no package in the list pulls it in.
CARD_PACKAGES = [
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "bitsandbytes==0.46.1",
    "librosa==0.10.2.post1",
    "soundfile==0.13.1",
    "vocos==0.1.0",
    "vector-quantize-pytorch==1.18.5",
    "Pillow",
]
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY] + CARD_PACKAGES)
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The model card's stack did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: card stack install failed in venv")

# 3c. flash-attn: the one package the sealed environment was missing.
# MiniCPM's own modeling file imports flash_attn, and transformers
# checks that file's imports before it loads a single weight. The
# fifth round's full-capture log (minicpm_load_log.txt) shows all
# three rungs dying in about three seconds on exactly that check:
# "This modeling file requires the following packages that were not
# found in your environment: flash_attn". Colab's stock environment
# carries flash-attn 2.8.3 already, which is why the model card never
# lists it; the private environment has to be given its own copy. It
# goes in as the official prebuilt wheel, never a build from source:
# Dao-AILab's flash-attn 2.8.3 for CPython 3.12, torch 2.8, and CUDA
# 12, in the cxx11 ABI TRUE form, which is the ABI torch 2.8 itself
# ships with (PyTorch dropped its pre-cxx11 ABI builds at 2.8). The
# address was fetched and the wheel opened when this round was
# built: it answers HTTP 200, it holds the cp312 CUDA extension, and
# its version reads 2.8.3. Its one extra dependency, einops, is
# small and comes from the normal shelf in the same pass.
FLASH_ATTN_WHEEL = ("https://github.com/Dao-AILab/flash-attention/"
                    "releases/download/v2.8.3/flash_attn-2.8.3"
                    "%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312"
                    "-linux_x86_64.whl")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 FLASH_ATTN_WHEEL])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The flash-attn wheel did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: flash-attn install failed in venv")

# 4. Proof, in a fresh process under the environment's Python. Version
# alone is not accepted as proof anymore: the proof also imports
# WHISPER_ATTENTION_CLASSES, the exact symbol MiniCPM's own code asks
# transformers for, whose absence is what a wrong transformers version
# actually breaks, and flash_attn, the package the model's modeling
# file demands at the door (see 3c above).
PROOF = (
    "import sys, torch, transformers\n"
    "print('venv python:', sys.version.split()[0])\n"
    "print('torch version:', torch.__version__)\n"
    "print('transformers version:', transformers.__version__)\n"
    "print('transformers file:', transformers.__file__)\n"
    "from transformers.models.whisper.modeling_whisper import "
    "WHISPER_ATTENTION_CLASSES\n"
    "print('WHISPER_ATTENTION_CLASSES import: OK', "
    "sorted(WHISPER_ATTENTION_CLASSES.keys()))\n"
    "import flash_attn\n"
    "print('flash_attn version:', flash_attn.__version__)\n"
)
proof = subprocess.run([VENV_PY, "-c", PROOF],
                       capture_output=True, text=True)
print()
print(proof.stdout)
if proof.stderr.strip():
    print("proof process also said:")
    print("\n".join(proof.stderr.splitlines()[-10:]))
ok = ("transformers version: 4.44.2" in proof.stdout
      and "WHISPER_ATTENTION_CLASSES import: OK" in proof.stdout
      and "flash_attn version: 2.8.3" in proof.stdout)
if not ok:
    print("!" * 72)
    print("STOP. THE PROOF FAILED INSIDE THE PRIVATE ENVIRONMENT.")
    print("Either a version is wrong (transformers 4.44.2, flash_attn")
    print("2.8.3) or one of the proof imports failed: the whisper symbol")
    print("or flash_attn. Do not go on to Step 2. Show this to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: venv proof failed")
print("proof passed: transformers 4.44.2, loaded from the path above,")
print("WHISPER_ATTENTION_CLASSES imports from it, and flash_attn")
print("2.8.3 imports too.")

# The chip, read from the machine itself. This cell deliberately never
# imports torch or transformers into the notebook's own session.
smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print()
print((smi.stdout or smi.stderr).strip())
print("This notebook wants an L4 (about 24 GB). A T4 (about 16 GB)")
print("cannot hold the model in full form; the ladder would have to")
print("squeeze it, and the record would say so.")
print("Step 1 done. Go on to Step 2. No restart is needed.")

## Step 2 · Get the codec battery files

Everything this run asks about lives in your Google Drive,
in the folder named `codec-control`. The next cell links
your Drive, finds that folder, and copies fifteen files to
a plain work folder here:

- Seven Arm A files, `codec_<sound>_aac.mp4`: the swap
  still picture carrying each sound, built with the swap
  recipe.
- Seven Arm B masters, `codec_<sound>_raw.wav`: the same
  sounds as uncompressed WAV, 16-bit PCM at 48000 Hz (the
  speech master is stereo, a byte-for-byte copy of the swap
  study's own speech track; the rest are mono).
- `codec_still.png`: the swap still itself, the picture
  the Arm B units carry.

The sounds, in battery order: `tone_ref`, `tone_quiet`,
`clicks_quiet`, `speech`, `tone_under_bed`, `burst_quiet`,
`silence`. Levels and hashes for every file are recorded in
the battery's BATTERY.md; the runner re-measures everything
from inside this session in Step 3 before it believes any
of it.

Do not go on until all fifteen files say "ready".

In [ ]:
# --- Get the codec battery files -------------------------------------------------
import os, shutil, subprocess

SOUNDS = ["tone_ref", "tone_quiet", "clicks_quiet", "speech",
          "tone_under_bed", "burst_quiet", "silence"]
FILES = ([f"codec_{s}_aac.mp4" for s in SOUNDS]
         + [f"codec_{s}_raw.wav" for s in SOUNDS]
         + ["codec_still.png"])
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/codec-control"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/codec-control"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

ready = True
for name in FILES:
    bring(name)
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        d = duration_s(path)
        extra = f", {d:.3f} s" if d is not None else ""
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.2f} MB{extra})")
    else:
        ready = False
        print("MISSING:", name)
if not ready:
    print()
    print("!" * 72)
    print("STOP. Some files are missing. Check the Drive folder name is")
    print("exactly codec-control, or upload the files by hand, then")
    print("run this cell again. Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2")
print()
print("ready: all 7 Arm A MP4s, all 7 Arm B WAVs, and the still")

## Step 3 · Write the runner, then check: ladder and file assertions

The first cell below writes the runner script,
`/content/minicpm_codec_run.py`. It holds the same proof and
the same loading ladder as the Series Two MiniCPM runner
(bf16, then 8-bit, then 4-bit, all pinned to revision
`06849bfd36da94da1f5a88fa17e8ba63f08a4c4d`, remote-code
cache cleared first), with rung tracebacks appended to
`codec_minicpm_load_log.txt` in the clips folder. What
changed is the battery and its gates.

The second cell runs the script in **check mode**: proof
lines (including the yes-score fact: `model.chat` exposes
no score, settled from the code), the ladder, and then the
battery file assertions. Every one of the fourteen battery
files is decoded through the very path its arm will ride,
printed with its SHA-256, duration, and per-second RMS, and
asserted: 10.0 seconds within half a second; sound where
sound is declared, true zeros where silence is declared;
and for the seven WAVs, the waveform handed toward the
model must hash equal to a direct decode of the same WAV.
Any failure stops the run here, exit code 7, before a
single ask.

There is deliberately no sanity question. The gates are the
files.

In [ ]:
# --- Write the codec control runner script --------------------------------------
# The runner holds the ladder, the battery file assertions,
# the strict feed proof, and the 56 codec asks. It is
# written to a plain file here and only ever run under the
# private environment's Python.
RUNNER_SOURCE = '\n# minicpm_codec_run.py - the codec-loss control runner for MiniCPM-o 2.6.\n# Written by project_k_minicpm_codec_colab.ipynb. Run only under the\n# private environment\'s Python (/content/minicpm-venv/bin/python).\n# Modes: "check" runs the proof, the loading ladder, and the battery\n# file assertions, then stops. "battery" does all of that and then\n# runs the 56 codec asks and saves the CSV.\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nfrom transformers.models.whisper.modeling_whisper import WHISPER_ATTENTION_CLASSES\nprint("WHISPER_ATTENTION_CLASSES import: OK",\n      sorted(WHISPER_ATTENTION_CLASSES.keys()))\nif transformers.__version__ != "4.44.2":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.44.2.")\n    sys.exit(3)\n\n# The yes-score question, settled from the model\'s own code before\n# the run (modeling_minicpmo.py in the pinned revision): MiniCPMO.chat\n# calls generate, _decode requests sequences and hidden states from\n# llm.generate but never output scores, and chat returns only the\n# decoded text. There is no score surface on the chat door, so this\n# constant is False and every CSV row marks the score as not exposed.\nYES_SCORE_EXPOSED = False\nprint("yes token score exposed by model.chat:", YES_SCORE_EXPOSED,\n      "(chat returns decoded text only; output scores are never requested)")\n\n# --- Load the model: the same MiniCPM-o ladder as Series One --------------------\nimport traceback, shutil, datetime\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\n\nREPO = "openbmb/MiniCPM-o-2_6"\nREVISION = "06849bfd36da94da1f5a88fa17e8ba63f08a4c4d"\n\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/codec-control/codec_minicpm_load_log.txt",\n                  "/content/codec_minicpm_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("MiniCPM-o 2.6 codec control load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\nlog_load("repo: " + REPO + " at pinned revision " + REVISION + "\\n")\n\ntry:\n    from transformers.dynamic_module_utils import HF_MODULES_CACHE\n    module_cache = os.path.join(str(HF_MODULES_CACHE), *REPO.split("/"))\n    if os.path.isdir(module_cache):\n        shutil.rmtree(module_cache)\n        clear_note = "cleared cached remote code for this repo: " + module_cache\n    else:\n        clear_note = ("no cached remote code for this repo to clear: "\n                      + module_cache)\nexcept Exception as e:\n    clear_note = "WARNING: the remote-code cache clear did not run: " + str(e)\nprint(clear_note)\nlog_load(clear_note + "\\n")\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\ntokenizer = None\nMODEL_NAME = None\nLOAD_MODE = None\n\ndef try_load(mode):\n    kwargs = dict(trust_remote_code=True, attn_implementation="sdpa",\n                  init_vision=True, init_audio=True, init_tts=False)\n    if mode == "bf16":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, **kwargs)\n        m = m.eval().cuda()\n    elif mode == "8-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, load_in_8bit=True,\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    elif mode == "4-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16,\n            quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    t = AutoTokenizer.from_pretrained(REPO, revision=REVISION,\n                                      trust_remote_code=True)\n    return m, t\n\nrung_errors = []\nfor mode in ("bf16", "8-bit", "4-bit"):\n    print("=" * 70)\n    print(f"rung: {REPO} in {mode} form")\n    try:\n        m, t = try_load(mode)\n        model, tokenizer = m, t\n        MODEL_NAME, LOAD_MODE = REPO, mode\n        print("loaded:", REPO, "in", mode, "form")\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        rung_errors.append((REPO, mode, tb))\n        print("this rung failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("rung failed: " + REPO + " in " + mode + " form\\n")\n        log_load(tb)\n        model = None\n        tokenizer = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("ALL THREE RUNGS FAILED. MiniCPM-o 2.6 did not load in any fit.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every rung is saved, word for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\ndef answer_text(res):\n    if isinstance(res, dict):\n        return str(res.get("text", "") or "")\n    if isinstance(res, (list, tuple)):\n        return answer_text(res[0]) if res else ""\n    return str(res or "")\n\n# --- Feed machinery -----------------------------------------------------------------\nimport csv, glob, hashlib, json, math, re, subprocess\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom PIL import Image\n\nCLIPS = "/content/clips"\nSTILL = os.path.join(CLIPS, "codec_still.png")\nTRACK_DIRS = ["/content/drive/MyDrive/codec-control/codec_extracted_tracks",\n              "/content/codec_extracted_tracks"]\nfor d in TRACK_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\nFEED_AAC = ("omni chat units: 1 frame per second + the matching 1s "\n            "of the 16 kHz audio track per unit, frames downscaled "\n            "to fit 448x448, max 15 units, max_slice_nums=1, "\n            "greedy (sampling=False), no generated speech; the "\n            "audio track was decoded out of the Arm A MP4 by "\n            "ffmpeg and loaded at 16 kHz, the Series Two path")\nFEED_RAW = ("omni chat units: the swap still (codec_still.png) + "\n            "the matching 1s of the 16 kHz waveform loaded "\n            "directly from the WAV file with librosa per unit, "\n            "frames downscaled to fit 448x448, max 15 units, "\n            "max_slice_nums=1, greedy (sampling=False), no "\n            "generated speech; the audio array never passes "\n            "through an MP4 or an AAC decode")\n\nclass FeedProofError(RuntimeError):\n    pass\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef media_duration(path):\n    out = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "csv=p=0", path], capture_output=True, text=True)\n    try:\n        return float(out.stdout.strip())\n    except Exception:\n        return None\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\ndef waveform_sha256(audio_np):\n    arr = np.ascontiguousarray(np.asarray(audio_np, dtype=np.float32))\n    return hashlib.sha256(arr.tobytes()).hexdigest()\n\nFRAME_DIR = "/content/codec_frames"\nos.makedirs(FRAME_DIR, exist_ok=True)\n_feed_cache = {}\n\ndef _proof_and_units(path, audio_np, sr, with_picture, frames,\n                     frame_rc, audio_rc, feed_text):\n    # Shared measurement and assertion core: Arm A calls it with\n    # the track extracted from the MP4, Arm B with the waveform\n    # loaded from the WAV. The checks are the Series Two ones,\n    # plus the battery\'s duration assertion.\n    src_dur = media_duration(path)\n    if src_dur is None:\n        raise FeedProofError(f"could not read media duration for {path}")\n    if abs(src_dur - 10.0) > 0.5:\n        raise FeedProofError(\n            f"battery file duration is not 10.0 s for {path}: "\n            f"{src_dur:.3f}s")\n    unit_count = int(round(src_dur))\n    if unit_count <= 0:\n        raise FeedProofError(f"media duration rounds to zero units: {path}")\n    if unit_count > 15:\n        unit_count = 15\n    track_dur = len(audio_np) / sr if sr else 0.0\n    if len(audio_np) == 0:\n        raise FeedProofError(f"track is empty for {path}")\n    if track_dur < unit_count - 0.5:\n        raise FeedProofError(\n            f"track is short for {path}: {track_dur:.3f}s of "\n            f"track for {unit_count} units")\n    if with_picture and frames and len(frames) < unit_count - 1:\n        raise FeedProofError(\n            f"frame extraction is short for {path}: {len(frames)} "\n            f"frames for {unit_count} units")\n\n    per_unit_rms = []\n    per_unit_padding = []\n    zero_chunks = 0\n    for i in range(unit_count):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) == 0:\n            raise FeedProofError(\n                f"unit {i} has no audio samples for {path}")\n        per_unit_rms.append(round(rms_db(chunk), 2))\n        if not np.any(chunk):\n            zero_chunks += 1\n        per_unit_padding.append(int(sr - len(chunk)) if len(chunk) < sr else 0)\n\n    overall_rms = round(rms_db(audio_np), 2)\n    nonzero_share = float(np.count_nonzero(audio_np) / len(audio_np))\n    return {"src_dur": src_dur, "unit_count": unit_count,\n            "track_dur": track_dur, "per_unit_rms": per_unit_rms,\n            "per_unit_padding": per_unit_padding,\n            "zero_chunks": zero_chunks, "overall_rms": overall_rms,\n            "nonzero_share": nonzero_share}\n\ndef _finish_feed(path, audio_np, sr, m, frame_rc, audio_rc,\n                 expected_sound, expected_silence, feed_text,\n                 contents):\n    if expected_sound and (m["nonzero_share"] == 0.0\n                           or m["overall_rms"] <= -119.5\n                           or m["zero_chunks"] == m["unit_count"]):\n        raise FeedProofError(\n            f"track that should carry sound decoded silent for {path}: "\n            f"rms={m[\'overall_rms\']} dB, "\n            f"nonzero share={m[\'nonzero_share\']}")\n    if expected_silence and m["nonzero_share"] > 0.001:\n        raise FeedProofError(\n            f"declared silence track is not silent for {path}: "\n            f"nonzero share={m[\'nonzero_share\']}")\n    track_hash = waveform_sha256(audio_np)\n    track_name = (re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n                  + "__" + track_hash[:12] + ".wav")\n    track_paths = []\n    for d in TRACK_DIRS:\n        try:\n            tp = os.path.join(d, track_name)\n            if not os.path.exists(tp):\n                sf.write(tp, audio_np, sr, subtype="PCM_16")\n            track_paths.append(tp)\n        except Exception:\n            pass\n    proof = {\n        "file": os.path.basename(path),\n        "source_file_sha256": file_sha256(path),\n        "frame_ffmpeg_rc": frame_rc,\n        "audio_ffmpeg_rc": audio_rc,\n        "track_duration_s": round(m["track_dur"], 3),\n        "sample_rate": int(sr),\n        "sample_count": int(len(audio_np)),\n        "track_rms_db": m["overall_rms"],\n        "track_nonzero_share": round(m["nonzero_share"], 6),\n        "unit_count": int(m["unit_count"]),\n        "per_unit_rms_db": m["per_unit_rms"],\n        "per_unit_padding_samples": m["per_unit_padding"],\n        "padding_samples_total": int(sum(m["per_unit_padding"])),\n        "zero_chunk_count": int(m["zero_chunks"]),\n        "track_sha256": track_hash,\n        "track_copy_paths": track_paths,\n        "expected_sound": bool(expected_sound),\n        "expected_silence": bool(expected_silence),\n        "feed": feed_text,\n    }\n    return {"contents": contents, "proof": proof}\n\ndef extract_feed(path, expected_sound, expected_silence):\n    # Arm A: the Series Two with-picture extraction, unchanged.\n    # Frames at 1 fps from the MP4; the soundtrack decoded out\n    # of the MP4 to a 16 kHz WAV and loaded with librosa; units\n    # of frame plus the matching second of sound.\n    key = (path, True, bool(expected_sound), bool(expected_silence))\n    if key in _feed_cache:\n        return _feed_cache[key]\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    tag = re.sub(r"[^A-Za-z0-9]+", "_",\n                 os.path.basename(path) + "_aac")\n    fdir = os.path.join(FRAME_DIR, tag)\n    os.makedirs(fdir, exist_ok=True)\n    for old in glob.glob(os.path.join(fdir, "*")):\n        os.remove(old)\n\n    fproc = subprocess.run(\n        ["ffmpeg", "-y", "-v", "error", "-i", path, "-vf", "fps=1",\n         os.path.join(fdir, "f%03d.png")], capture_output=True, text=True)\n    frame_rc = fproc.returncode\n    frames = sorted(glob.glob(os.path.join(fdir, "*.png")))\n    if frame_rc != 0:\n        raise FeedProofError(\n            f"frame extraction failed for {path}: rc={frame_rc}: "\n            + (fproc.stderr or "")[-500:])\n    if not frames:\n        raise FeedProofError(f"frame extraction produced no frames: {path}")\n\n    wav = os.path.join(fdir, "audio.wav")\n    aproc = subprocess.run(\n        ["ffmpeg", "-y", "-v", "error", "-i", path, "-vn", "-ac", "1",\n         "-ar", "16000", wav], capture_output=True, text=True)\n    audio_rc = aproc.returncode\n    if audio_rc != 0:\n        raise FeedProofError(\n            f"audio extraction failed for {path}: rc={audio_rc}: "\n            + (aproc.stderr or "")[-500:])\n    if not os.path.exists(wav):\n        raise FeedProofError(f"audio extraction produced no wav for {path}")\n    audio_np, sr = librosa.load(wav, sr=16000, mono=True)\n    audio_np = np.asarray(audio_np, dtype=np.float32)\n\n    m = _proof_and_units(path, audio_np, sr, True, frames,\n                         frame_rc, audio_rc, FEED_AAC)\n    contents = []\n    for i in range(m["unit_count"]):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) < sr:\n            chunk = np.pad(chunk, (0, sr - len(chunk))).astype(np.float32)\n        img = Image.open(frames[min(i, len(frames) - 1)]).convert("RGB")\n        img.thumbnail((448, 448))\n        contents.extend(["<unit>", img, chunk])\n    feed = _finish_feed(path, audio_np, sr, m, frame_rc, audio_rc,\n                        expected_sound, expected_silence, FEED_AAC,\n                        contents)\n    _feed_cache[key] = feed\n    return feed\n\ndef extract_feed_raw(path, expected_sound, expected_silence):\n    # Arm B: the waveform is loaded directly from the WAV file\n    # with librosa at 16 kHz mono. No MP4 is opened and no\n    # ffmpeg call touches the audio. The units carry the swap\n    # still (codec_still.png) and the matching second of that\n    # waveform, in the same shape as Arm A\'s units.\n    key = (path, False, bool(expected_sound), bool(expected_silence))\n    if key in _feed_cache:\n        return _feed_cache[key]\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    if not os.path.exists(STILL):\n        raise FeedProofError(f"the still image is missing: {STILL}")\n    audio_np, sr = librosa.load(path, sr=16000, mono=True)\n    audio_np = np.asarray(audio_np, dtype=np.float32)\n    frame_rc = ("n/a (the picture is the still image file "\n                "codec_still.png, opened directly; this path "\n                "makes no ffmpeg call)")\n    audio_rc = ("n/a (the waveform is loaded from the WAV "\n                "file with librosa; this path makes no ffmpeg "\n                "call and opens no MP4)")\n    # frames=None: the picture is one still reused for every\n    # unit, so the frame-count check (which guards Arm A\'s\n    # per-second frame extraction) does not apply here.\n    m = _proof_and_units(path, audio_np, sr, True, None,\n                         frame_rc, audio_rc, FEED_RAW)\n    contents = []\n    for i in range(m["unit_count"]):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) < sr:\n            chunk = np.pad(chunk, (0, sr - len(chunk))).astype(np.float32)\n        img = Image.open(STILL).convert("RGB")\n        img.thumbnail((448, 448))\n        contents.extend(["<unit>", img, chunk])\n    feed = _finish_feed(path, audio_np, sr, m, frame_rc, audio_rc,\n                        expected_sound, expected_silence, FEED_RAW,\n                        contents)\n    _feed_cache[key] = feed\n    return feed\n\ndef ask(job, prompt, max_new_tokens):\n    path = os.path.join(CLIPS, job["file"])\n    if job["arm"] == "raw_wav":\n        feed = extract_feed_raw(path, job["expected_sound"],\n                                job["expected_silence"])\n    else:\n        feed = extract_feed(path, job["expected_sound"],\n                            job["expected_silence"])\n    proof = feed["proof"]\n    print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n    contents = list(feed["contents"]) + [prompt]\n    sys_msg = model.get_sys_prompt(mode="omni", language="en")\n    res = model.chat(msgs=[sys_msg, {"role": "user", "content": contents}],\n                     tokenizer=tokenizer, sampling=False,\n                     max_new_tokens=max_new_tokens, omni_input=True,\n                     use_tts_template=False, generate_audio=False,\n                     max_slice_nums=1, use_image_id=False)\n    text = answer_text(res).strip()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return text, proof\n\n# --- The battery -------------------------------------------------------------------\n# The job list was built and counted by the notebook\'s builder\n# (7 sounds x 2 arms x 2 forms x 2 repeats = 56 asks) and is\n# injected here as a literal. The open prompt is the swap\n# study\'s P3, pulled from pilot2/swap/run_swap.py at build\n# time; the yes/no prompt is the Series Two template with the\n# codec targets.\nJOBS = [({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16)]\n\nprint(f"codec battery: {len(JOBS)} asks "\n      f"(7 sounds x 2 arms x 2 forms x 2 repeats)")\n\nFIELDS = ["test", "ask_id", "sound", "file", "arm", "version",\n          "question_form", "question_label", "target", "prompt",\n          "repeat", "target_start_s", "expected_sound",\n          "expected_silence", "frame_ffmpeg_rc", "audio_ffmpeg_rc",\n          "track_duration_s", "sample_rate", "sample_count",\n          "track_rms_db", "track_nonzero_share", "unit_count",\n          "per_unit_rms_db", "per_unit_padding_samples",\n          "padding_samples_total", "zero_chunk_count",\n          "track_sha256", "track_copy_paths", "source_file_sha256",\n          "raw_answer", "yes_score", "yes_score_status", "model",\n          "feed"]\n\nOUT_PATHS = ["/content/drive/MyDrive/codec-control/codec_minicpm_results.csv",\n             "/content/codec_minicpm_results.csv"]\n\ndef ask_id_of(job, form, target, rep):\n    return (f"{job[\'sound\']}|{job[\'arm\']}|{form}|{target}|{rep}")\n\n# --- Battery file assertions: the only gates ---------------------------------------\n# Before any ask, every battery file is decoded through the\n# very path its arm will ride and asserted. For Arm B the\n# waveform handed toward the model must also hash equal to\n# an independent direct decode of the same WAV in this same\n# session. A failure here stops the run with exit code 7\n# before a single question is asked. There is no\n# model-answer sanity gate in this runner: a past gate\n# misfired on a missing file, so the gates are the files\n# themselves.\ndef assert_battery_files():\n    seen = {}\n    for job, form, label, target, prompt, rep, max_tokens in JOBS:\n        key = (job["file"], job["arm"])\n        if key in seen:\n            continue\n        seen[key] = True\n        path = os.path.join(CLIPS, job["file"])\n        print("-" * 70)\n        print(f"FILE ASSERT {job[\'arm\']}: {job[\'file\']}")\n        try:\n            if job["arm"] == "raw_wav":\n                feed = extract_feed_raw(path, job["expected_sound"],\n                                        job["expected_silence"])\n                proof = dict(feed["proof"])\n                canon, csr = librosa.load(path, sr=16000, mono=True)\n                canon_hash = waveform_sha256(canon)\n                if canon_hash != proof["track_sha256"]:\n                    raise FeedProofError(\n                        f"Arm B waveform hash mismatch for {path}: "\n                        f"the feed carries {proof[\'track_sha256\']}, "\n                        f"a direct decode of the WAV gives "\n                        f"{canon_hash}")\n                proof["wav_canonical_sha256"] = canon_hash\n                native, nsr = sf.read(path, dtype="float32",\n                                      always_2d=True)\n                native_mono = native.mean(axis=1)\n                proof["wav_native_sample_rate"] = int(nsr)\n                proof["wav_native_rms_db"] = round(\n                    rms_db(native_mono), 2)\n                proof["wav_native_duration_s"] = round(\n                    len(native_mono) / nsr, 3)\n            else:\n                feed = extract_feed(path, job["expected_sound"],\n                                    job["expected_silence"])\n                proof = feed["proof"]\n        except FeedProofError as e:\n            print()\n            print("!" * 70)\n            print("BATTERY FILE ASSERTION FAILED:")\n            print(str(e))\n            print("No ask has been taken. Fix the file set and run")\n            print("again. Show this whole output to Muse.")\n            print("!" * 70)\n            sys.exit(7)\n        print("FILE PROOF " + json.dumps(proof, sort_keys=True))\n    print()\n    print(f"file assertions passed for {len(seen)} files "\n          f"(7 sounds x 2 arms).")\n\nassert_battery_files()\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, ladder, and file assertions")\n    print("all passed. The battery was not run. Run Step 4 for the")\n    print("56 asks.")\n    sys.exit(0)\n\n# --- Resume: load any rows already saved --------------------------------------------\nrows = []\ndone_keys = set()\nfor p in OUT_PATHS:\n    if os.path.exists(p):\n        try:\n            with open(p, newline="") as fh:\n                rows = list(csv.DictReader(fh))\n            print(f"resume: loaded {len(rows)} existing rows from {p}")\n            break\n        except Exception as e:\n            print("resume: could not read", p, "-", str(e)[:120])\n            rows = []\nfor r in rows:\n    done_keys.add((r.get("sound", ""), r.get("arm", ""),\n                   r.get("question_form", ""), r.get("target", ""),\n                   str(r.get("repeat", ""))))\nfailed = 0\n\ndef save_rows():\n    for out_path in OUT_PATHS:\n        try:\n            with open(out_path, "w", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=FIELDS)\n                w.writeheader()\n                w.writerows(rows)\n            print("saved:", out_path)\n        except Exception as e:\n            print("could not save", out_path, "-", str(e)[:120])\n\nfor job, form, label, target, prompt, rep, max_tokens in JOBS:\n    key = (job["sound"], job["arm"], form, target, str(rep))\n    aid = ask_id_of(job, form, target, rep)\n    if key in done_keys:\n        print("=" * 70)\n        print(f"SKIP (already in the CSV): {aid}")\n        continue\n    print("=" * 70)\n    print(f"codec_control - sound={job[\'sound\']} arm={job[\'arm\']} "\n          f"version={job[\'version\']} form={form}/{label} "\n          f"target={target} repeat={rep}")\n    raw_text = ""\n    error = ""\n    proof = None\n    try:\n        raw, proof = ask(job, prompt, max_tokens)\n        print(f"ASK {aid} arm={job[\'arm\']} file={job[\'file\']} "\n              f"waveform_rms_db={proof[\'track_rms_db\']}")\n        if not raw.strip():\n            error = "ERROR: empty answer returned"\n            failed += 1\n            print("WARNING: this ask came back EMPTY. Recorded as an error,")\n            print("never as silence.")\n            raw_text = error\n        else:\n            raw_text = raw\n            print("RAW ANSWER:")\n            print(raw)\n    except FeedProofError as e:\n        print()\n        print("!" * 70)\n        print("FEED PROOF HARD STOP. The feed for this ask failed its proof:")\n        print(str(e))\n        print("No answer was taken for this ask, and the run stops here.")\n        print("Rows completed before this ask are saved now. Show this")\n        print("whole output to Muse.")\n        print("!" * 70)\n        save_rows()\n        sys.exit(7)\n    except Exception as e:\n        error = "ERROR: " + traceback.format_exc()\n        failed += 1\n        print(f"WARNING: this ask FAILED with an error. The full error is")\n        print("saved in the CSV column raw_answer.")\n        raw_text = error\n        proof = proof or {}\n    p = proof or {}\n    rows.append({\n        "test": job["test"], "ask_id": aid, "sound": job["sound"],\n        "file": job["file"], "arm": job["arm"],\n        "version": job["version"], "question_form": form,\n        "question_label": label, "target": target, "prompt": prompt,\n        "repeat": rep, "target_start_s": job.get("target_start_s", ""),\n        "expected_sound": job["expected_sound"],\n        "expected_silence": job["expected_silence"],\n        "frame_ffmpeg_rc": p.get("frame_ffmpeg_rc", ""),\n        "audio_ffmpeg_rc": p.get("audio_ffmpeg_rc", ""),\n        "track_duration_s": p.get("track_duration_s", ""),\n        "sample_rate": p.get("sample_rate", ""),\n        "sample_count": p.get("sample_count", ""),\n        "track_rms_db": p.get("track_rms_db", ""),\n        "track_nonzero_share": p.get("track_nonzero_share", ""),\n        "unit_count": p.get("unit_count", ""),\n        "per_unit_rms_db": json.dumps(p.get("per_unit_rms_db", "")),\n        "per_unit_padding_samples": json.dumps(\n            p.get("per_unit_padding_samples", "")),\n        "padding_samples_total": p.get("padding_samples_total", ""),\n        "zero_chunk_count": p.get("zero_chunk_count", ""),\n        "track_sha256": p.get("track_sha256", ""),\n        "track_copy_paths": json.dumps(p.get("track_copy_paths", "")),\n        "source_file_sha256": p.get("source_file_sha256", ""),\n        "raw_answer": raw_text,\n        "yes_score": "not exposed",\n        "yes_score_status": ("model.chat returns decoded text only and "\n                             "does not request output scores"),\n        "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                  "[venv py3.12, torch 2.8.0+cu126]"),\n        "feed": p.get("feed", ""),\n    })\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed or came back empty. Their")\n    print("raw words or full errors are in the CSV column raw_answer.")\n    print("This run is NOT clean evidence until those asks are explained.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered, every feed proved in print first.")\n\nsave_rows()\nif failed:\n    sys.exit(6)\n'
with open("/content/minicpm_codec_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/minicpm_codec_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, ladder, file assertions -------------------------------------
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_codec_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a ladder rung, or a battery file")
    print("assertion. Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, every battery")
print("file asserted on its real feed path. Go on to Step 4.")

## Step 4 · Run the codec battery

The next cell runs the runner in **battery mode**: proof,
ladder, and the file assertions again, then the 56 asks.
Every ask prints its FEED PROOF line before the model's
answer is taken, and an ASK line naming the ask, its arm,
its file, and the RMS of the waveform actually passed.

**How the arms differ, stated plainly.** Arm A rides the
Series Two extraction: frames at one per second out of the
MP4, its soundtrack decoded to 16 kHz, one-second units of
frame plus sound chunk through the omni chat. Arm B loads
the waveform straight from the WAV at 16 kHz and builds the
same units, each carrying the swap still and the matching
second of that waveform. The picture is the same still in
both arms. The sound content is the same sound in both
arms. The only difference is what the sound passed through
on its way in: an AAC encode and decode, or nothing.

**Token caps.** 300 for the open question, 16 for yes/no.
Decoding is greedy everywhere (sampling=False).

**Hard stops and warnings.** A feed that fails its proof
stops the whole run on the spot (exit code 7), with the
rows completed so far saved first. An ask that errors or
comes back empty is saved with its full error text in
`raw_answer` and a WARNING on screen, never as silence,
and the run ends with the RUN PROBLEM banner (exit code
6). If the session dies, run Step 4 again: asks already
in `codec_minicpm_results.csv` are skipped.

Results save to `codec_minicpm_results.csv` in the Drive
clips folder, one row per ask, with the full feed-proof
numbers on every row, in the Series Two schema plus an
ask_id column. The yes-score column reads "not exposed" on
every row, as in this model's Series Two run.

In [ ]:
# --- Battery mode: the codec control asks --------------------------------------------
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_codec_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("or some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered, every feed proved, CSV saved.")

## Closing · How to read the answers

Read the two arms against each other, sound by sound,
before reading sounds against each other. The reading was
fixed in advance:

- **Caught raw, missed in AAC:** the file did it. The
  codec, not the model, lost that sound, and any earlier
  Project K miss on a sound like it has to be re-labeled.
- **Missed in both:** the model did it. The sound arrived
  intact (the Arm B proof says so, hash and all) and was
  still not reported. The codec objection dies for that
  sound.
- **Caught in both:** codec loss does not explain that
  sound.
- **Caught in AAC, missed raw:** a build bug, by definition
  of the design. Stop and inspect the feed paths before
  believing any row.

Then compare with this model's Series Two record. In Series
Two, MiniCPM transcribed the speech perfectly through
another door and still reported hearing nothing; the swap
speech here is that same soundtrack, so its two arms speak
directly to whether any part of that reporting loss was
the file's. The silence rows matter in the other direction:
a yes on silence in either arm is a false alarm, and the
arms should agree on zeros, because both proofs verify the
track really is zeros.

- **Scope.** Every claim stays capped at this model,
  these sounds, these two feed paths, this date. Arm B
  proves what arrives when nothing is compressed away; it
  does not prove what any deployed system hears, because
  deployed systems ride files, which is Arm A's world, and
  that is exactly why both arms are in the record.

When you finish, the file `codec_minicpm_results.csv` in
your Drive is the record. Send it back, and the codec
control gets written up next to the Series Two runs.